In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import numpy as np
import pandas as pd
import os
import pickle
import gc
# 分布確認
# import pandas_profiling as pdp → import ydata_profiling as pdp
# 可視化
import matplotlib.pyplot as plt
# 分布確認
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder, OneHotEncoder
# モデリング
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix
import lightgbm as lgb

import warnings
warnings.filterwarnings('ignore')

# !pip install japanize-matplotlib
# import japanize_matplotlib
%matplotlib inline

In [ ]:
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
df_train.head()

In [ ]:
x_train, y_train, id_train = df_train[['Pclass', 'Fare']], df_train[['Survived']], df_train[['PassengerId']]
print(x_train.shape, y_train.shape, id_train.shape)

In [ ]:
import optuna

In [ ]:
# 探索しないハイパーパラメータ
params_base = {
    'boosting_type': 'gbdt',
    'objective': 'binary',
    'metric': 'auc',
    'learning_rate': 0.02,
    'n_estimators': 100000,
    'bagging_freq': 1,
    'seed': 123,
}

def objective(trial):
    # 探索するハイパーパラメータ
    params_tuning = {
        'num_leaves': trial.suggest_int('num_leaves', 8, 256),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 200),
        'min_sum_hessian_in_leaf': trial.suggest_float('min_sum_hessian_in_leaf', 1e-5, 1e-2, log=True),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 1.0),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-2, 1e2, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-2, 1e2, log=True),
    }
    params_tuning.update(params_base)

    list_metrics = []
    cv = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=123).split(x_train, y_train))
    for nfold in np.arange(5):
        idx_tr, idx_va = cv[nfold][0], cv[nfold][1]
        x_tr, y_tr = x_train.loc[idx_tr,:], y_train.loc[idx_tr,:]
        x_va, y_va = x_train.loc[idx_va,:], y_train.loc[idx_va,:]
        model = lgb.LGBMClassifier(**params_tuning)
        model.fit(x_tr, y_tr, eval_set=[(x_tr,y_tr), (x_va,y_va)], 
                  callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=True)],
                 )
        y_va_pred = model.predict_proba(x_va)[:,1]
        metrics_va = accuracy_score(y_va, np.where(y_va_pred>=0.5, 1, 0))
        list_metrics.append(metrics_va)

    metrics = np.mean(list_metrics)
    return metrics

In [ ]:
sampler = optuna.samplers.TPESampler(seed=123)
study = optuna.create_study(sampler=sampler, direction='maximize')
study.optimize(objective, n_trials=30)

In [ ]:
trial = study.best_trial
print('acc(best)={:.4f}'.format(trial.value))
display(trial.params)

In [ ]:
params_best = trial.params
params_best.update(params_base)
display(params_best)

In [ ]:
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

x_train = df_train[['Pclass', 'Age', 'Embarked']]
y_train = df_train[['Survived']]

In [ ]:
x_train['Age'] = x_train['Age'].fillna(x_train['Age'].mean())
x_train['Embarked'] = x_train['Embarked'].fillna(x_train['Embarked'].mode()[0])

In [ ]:
ohe = OneHotEncoder()
ohe.fit(x_train[['Embarked']])
df_embarked = pd.DataFrame(ohe.transform(x_train[['Embarked']]).toarray(), columns=['Embarked_{}'.format(col) for col in ohe.categories_[0]])

x_train = pd.concat([x_train, df_embarked], axis=1)
x_train = x_train.drop(columns=['Embarked'])

In [ ]:
x_train.head()

In [ ]:
x_train['Pclass'] = (x_train['Pclass'] - x_train['Pclass'].min()) / (x_train['Pclass'].max() - x_train['Pclass'].min())
x_train['Age'] = (x_train['Age'] - x_train['Age'].min()) / (x_train['Age'].max() - x_train['Age'].min())

In [ ]:
x_tr, x_va, y_tr, y_va = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train, random_state=123)
print(x_tr.shape, x_va.shape, y_tr.shape, y_va.shape)

In [ ]:
from sklearn.linear_model import LogisticRegression
model_logis = LogisticRegression()

model_logis.fit(x_tr, y_tr)

y_va_pred = model_logis.predict(x_va)
print('accuracy:{:.4f}'.format(accuracy_score(y_va, y_va_pred)))
print(y_va_pred[:5])

In [ ]:
y_va_pred_proba = model_logis.predict_proba(x_va)
print(y_va_pred_proba[:5,:])

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.layers import Embedding, Flatten, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau, LearningRateScheduler
from tensorflow.keras.optimizers import Adam, SGD

In [ ]:
def seed_everything(seed):
    import random
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    session_conf = tf.compat.v1.ConfigProto(
        intra_op_parallelism_threads=1,
        inter_op_parallelism_threads=1
    )
    sess = tf.compat.v1.Session(graph=tf.compat.v1.get_default_graph(),config=session_conf)
    tf.compat.v1.keras.backend.set_session(sess)

In [ ]:
df_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')

x_train = df_train[['Pclass', 'Age', 'Embarked']]
y_train = df_train[['Survived']]

In [ ]:
x_train['Age'] = x_train['Age'].fillna(x_train['Age'].mean())

for col in ['Pclass', 'Age']:
    value_min = x_train[col].min()
    value_max = x_train[col].max()
    x_train[col] = (x_train[col] - value_min) / (value_max - value_min)

In [ ]:
x_train['Embarked'] = x_train['Embarked'].fillna(x_train['Embarked'].mode()[0])

ohe = OneHotEncoder()
ohe.fit(x_train[['Embarked']])
df_embarked = pd.DataFrame(ohe.transform(x_train[['Embarked']]).toarray(),
                          columns =['Embarked_{}'.format(col) for col in ohe.categories_[0]])
x_train = pd.concat([x_train.drop(columns=['Embarked']), df_embarked], axis=1)

In [ ]:
x_tr, x_va, y_tr, y_va = train_test_split(x_train, y_train, test_size=0.2, stratify=y_train, random_state=123)

In [ ]:
x_tr.head()

In [ ]:
def create_model():
    input_num = Input(shape=(5,))
    x_num = Dense(10, activation='relu')(input_num)
    x_num = BatchNormalization()(x_num)
    x_num = Dropout(0.3)(x_num)
    x_num = Dense(10, activation='relu')(x_num)
    x_num = BatchNormalization()(x_num)
    x_num = Dropout(0.2)(x_num)
    x_num = Dense(5, activation='relu')(x_num)
    x_num = BatchNormalization()(x_num)
    x_num = Dropout(0.1)(x_num)
    out = Dense(1, activation='sigmoid')(x_num)

    model = Model(inputs=input_num, outputs=out,)

    model.compile(optimizer='Adam',loss='binary_crossentropy', metrics=['binary_crossentropy'])
    return model

model = create_model()
model.summary()